In [56]:
import xml.etree.ElementTree as ET
import re
import unicodedata
import math
import sys
import spacy
import pandas as pd
import numpy as np
import nltk

from IPython.display import display
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from pathlib import Path
from rank_bm25 import BM25Okapi
from gensim import corpora, models, similarities

stemmer = PorterStemmer()
for recurso in ["punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)

/home/lorena/python/NLP-ml/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/lorena/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


# Metricas de Evaluacion

In [57]:
def tabla_pruebas(nombre_funcion, pruebas):
    resultados = []

    for prueba in pruebas:
        resultado = prueba["funcion"](*prueba["argumentos"])

        resultados.append({
            "Caso": prueba["caso"],
            "Entrada": prueba["entrada"],
            "Resultado": resultado
        })

    tabla = pd.DataFrame(resultados)

    display(
        tabla.style.set_caption(nombre_funcion)
    )

    return tabla

## Precision y Precision at K

In [58]:
def precision(re:list):
    result = 0.0
    size = len(re)
    if size > 0:
        relevance =  np.array(re)
        result = (np.sum(relevance == 1))/size
    return result

def precision_at_k(re:list, k:int):
    result = 0.0
    size = len(re)
    if size > 0 and k > 0 and size >= k:
        relevance = np.array(re)
        result = np.sum(relevance[:k] ==1)/k
    return result        

### Examples

In [59]:
pruebas_precision = [
    {
        "caso": "Ejemplo",
        "entrada": [0, 1, 0, 0, 1],
        "argumentos": [[0, 1, 0, 0, 1]],
        "funcion": precision
    },
    {
        "caso": "Prueba 1",
        "entrada": [1, 1, 0, 1, 0],
        "argumentos": [[1, 1, 0, 1, 0]],
        "funcion": precision
    },
    {
        "caso": "Prueba 2",
        "entrada": [1, 1, 1, 1],
        "argumentos": [[1, 1, 1, 1]],
        "funcion": precision
    },
    {
        "caso": "Vector vacío",
        "entrada": [],
        "argumentos": [[]],
        "funcion": precision
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": [0, 0, 0, 0],
        "argumentos": [[0, 0, 0, 0]],
        "funcion": precision
    }
]

tabla_precision = tabla_pruebas(
    "Precision",
    pruebas_precision
)

,Caso,Entrada,Resultado
0,Ejemplo,"[0, 1, 0, 0, 1]",0.400000
1,Prueba 1,"[1, 1, 0, 1, 0]",0.600000
2,Prueba 2,"[1, 1, 1, 1]",1.000000
3,Vector vacío,[],0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0]",0.000000


In [60]:
pruebas_precision_k = [
    {
        "caso": "Ejemplo",
        "entrada": "[0, 1, 0, 0, 1], k=3",
        "argumentos": [[0, 1, 0, 0, 1], 3],
        "funcion": precision_at_k
    },
    {
        "caso": "Prueba 1",
        "entrada": "[1, 1, 0, 1, 0], k=3",
        "argumentos": [[1, 1, 0, 1, 0], 3],
        "funcion": precision_at_k
    },
    {
        "caso": "Prueba 2",
        "entrada": "[0, 1, 1, 0, 1], k=4",
        "argumentos": [[0, 1, 1, 0, 1], 4],
        "funcion": precision_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], k=3",
        "argumentos": [[], 3],
        "funcion": precision_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[1, 0, 1], k=10",
        "argumentos": [[1, 0, 1], 10],
        "funcion": precision_at_k
    }
]

tabla_precision_k = tabla_pruebas(
    "Precision@K",
    pruebas_precision_k
)

,Caso,Entrada,Resultado
0,Ejemplo,"[0, 1, 0, 0, 1], k=3",0.333333
1,Prueba 1,"[1, 1, 0, 1, 0], k=3",0.666667
2,Prueba 2,"[0, 1, 1, 0, 1], k=4",0.500000
3,Vector vacío,"[], k=3",0.000000
4,K mayor que longitud,"[1, 0, 1], k=10",0.000000


## Recall at K

In [61]:
def recall_at_k(relevance_query:list, number_relevant_docs:int, k:int):
    result = 0.0
    size = len(relevance_query)
    if size > 0 and number_relevant_docs > 0 and k > 0 and size >= k:
        relevance = np.array(relevance_query)
        result = np.sum(relevance[:k] == 1)/number_relevant_docs
    return result

### Examples

In [62]:
pruebas_recall_k = [
    {
        "caso": "Ejemplo",
        "entrada": "[0, 1, 0, 0, 1], number_relevant_docs=4, k=3",
        "argumentos": [[0, 1, 0, 0, 1], 4, 3],
        "funcion": recall_at_k
    },
    {
        "caso": "Prueba 1",
        "entrada": "[1, 0, 1, 1, 0], number_relevant_docs=4, k=5",
        "argumentos": [[1, 0, 1, 1, 0], 4, 5],
        "funcion": recall_at_k
    },
    {
        "caso": "Prueba 2",
        "entrada": "[1, 0, 0, 1, 0], number_relevant_docs=5, k=5",
        "argumentos": [[1, 0, 0, 1, 0], 5, 5],
        "funcion": recall_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], number_relevant_docs=0, k=3",
        "argumentos": [[], 0, 3],
        "funcion": recall_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[1, 0, 1], number_relevant_docs=2, k=10",
        "argumentos": [[1, 0, 1], 2, 10],
        "funcion": recall_at_k
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0], number_relevant_docs=0, k=4",
        "argumentos": [[0, 0, 0, 0], 0, 4],
        "funcion": recall_at_k
    }
]

tabla_recall_k = tabla_pruebas(
    "Recall@K",
    pruebas_recall_k
)

,Caso,Entrada,Resultado
0,Ejemplo,"[0, 1, 0, 0, 1], number_relevant_docs=4, k=3",0.250000
1,Prueba 1,"[1, 0, 1, 1, 0], number_relevant_docs=4, k=5",0.750000
2,Prueba 2,"[1, 0, 0, 1, 0], number_relevant_docs=5, k=5",0.400000
3,Vector vacío,"[], number_relevant_docs=0, k=3",0.000000
4,K mayor que longitud,"[1, 0, 1], number_relevant_docs=2, k=10",0.000000
5,Sin documentos relevantes,"[0, 0, 0, 0], number_relevant_docs=0, k=4",0.000000


## Average precision

In [63]:
def average_precision(re: list):
    result = 0.0
    relevance = np.array(re)
    precisions=[]
    relevant_count = 0
    if len(relevance) > 0 and np.sum(relevance ==1) > 0 :
        for k, value in enumerate(relevance):
            if  value==1:
                relevant_count += 1
                precisions.append(relevant_count/(k+1))
        result = sum(precisions)/len(precisions)
    return result

### Examples

In [64]:
pruebas_average_precision = [
    {
        "caso": "Ejemplo",
        "entrada": "[1, 0, 1, 1, 0]",
        "argumentos": [[1, 0, 1, 1, 0]],
        "funcion": average_precision
    },
    {
        "caso": "Prueba 1",
        "entrada": "[0, 1, 0, 1, 1]",
        "argumentos": [[0, 1, 0, 1, 1]],
        "funcion": average_precision
    },
    {
        "caso": "Prueba 2",
        "entrada": "[1, 1, 0, 0, 1]",
        "argumentos": [[1, 1, 0, 0, 1]],
        "funcion": average_precision
    },
    {
        "caso": "Vector vacío",
        "entrada": "[]",
        "argumentos": [[]],
        "funcion": average_precision
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0, 0]",
        "argumentos": [[0, 0, 0, 0, 0]],
        "funcion": average_precision
    }
]

tabla_average_precision = tabla_pruebas(
    "Average Precision",
    pruebas_average_precision
)

,Caso,Entrada,Resultado
0,Ejemplo,"[1, 0, 1, 1, 0]",0.805556
1,Prueba 1,"[0, 1, 0, 1, 1]",0.533333
2,Prueba 2,"[1, 1, 0, 0, 1]",0.866667
3,Vector vacío,[],0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0, 0]",0.000000


## Mean average precision (MAP)

In [65]:
def mean_average_precision(relevance_queries:list[list]):
    precisions=[]
    result=0.0
    if len(relevance_queries) > 0:
        precisions = [average_precision(query) for query in relevance_queries]
        result = sum(precisions)/len(precisions)
    return result

### Examples

In [66]:
pruebas_map = [
    {
        "caso": "Ejemplo",
        "entrada": "[[1, 0, 1, 1, 0], [1, 1, 0, 0, 1]]",
        "argumentos": [
            [
                [1, 0, 1, 1, 0],
                [1, 1, 0, 0, 1]
            ]
        ],
        "funcion": mean_average_precision
    },
    {
        "caso": "Prueba 1",
        "entrada": "[[1, 1, 0, 1, 0], [0, 1, 1, 0, 0], [1, 0, 0, 1, 1]]",
        "argumentos": [
            [
                [1, 1, 0, 1, 0],
                [0, 1, 1, 0, 0],
                [1, 0, 0, 1, 1]
            ]
        ],
        "funcion": mean_average_precision
    },
    {
        "caso": "Prueba 2",
        "entrada": "[[1, 0, 0, 1], [1, 1, 0, 0], [0, 1, 1, 0]]",
        "argumentos": [
            [
                [1, 0, 0, 1],
                [1, 1, 0, 0],
                [0, 1, 1, 0]
            ]
        ],
        "funcion": mean_average_precision
    },
    {
        "caso": "Lista de consultas vacía",
        "entrada": "[]",
        "argumentos": [[]],
        "funcion": mean_average_precision
    },
    {
        "caso": "Consulta sin documentos relevantes",
        "entrada": "[[1, 0, 1, 0], [0, 0, 0, 0]]",
        "argumentos": [
            [
                [1, 0, 1, 0],
                [0, 0, 0, 0]
            ]
        ],
        "funcion": mean_average_precision
    }
]

tabla_map = tabla_pruebas(
    "Mean Average Precision (MAP)",
    pruebas_map
)

,Caso,Entrada,Resultado
0,Ejemplo,"[[1, 0, 1, 1, 0], [1, 1, 0, 0, 1]]",0.836111
1,Prueba 1,"[[1, 1, 0, 1, 0], [0, 1, 1, 0, 0], [1, 0, 0, 1, 1]]",0.733333
2,Prueba 2,"[[1, 0, 0, 1], [1, 1, 0, 0], [0, 1, 1, 0]]",0.777778
3,Lista de consultas vacía,[],0.000000
4,Consulta sin documentos relevantes,"[[1, 0, 1, 0], [0, 0, 0, 0]]",0.416667


## DCG at K

In [67]:
def dcg_at_k(relevance_query: list, k: int, gain: str):
    relevance = np.array(relevance_query)
    size = len(relevance)
    result = 0.0
    if size > 0 and k > 0 and size >= k:
        for i, rel in enumerate(relevance[:k]):
            if gain == 'linear':
                result += rel / math.log2(i + 2)
            elif gain == 'exponential':
                result += (2 ** rel - 1) / math.log2(i + 2)
    return result


### Example

In [68]:
pruebas_dcg_k = [
    {
        "caso": "Ejemplo 1 - ganancia lineal",
        "entrada": "[3, 2, 0, 1, 2], k=5, gain='linear'",
        "argumentos": [[3, 2, 0, 1, 2], 5, "linear"],
        "funcion": dcg_at_k
    },
    {
        "caso": "Ejemplo 2 - ganancia exponencial",
        "entrada": "[3, 0, 2, 1, 3], k=5, gain='exponential'",
        "argumentos": [[3, 0, 2, 1, 3], 5, "exponential"],
        "funcion": dcg_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], k=5, gain='linear'",
        "argumentos": [[], 5, "linear"],
        "funcion": dcg_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[3, 2, 1], k=10, gain='linear'",
        "argumentos": [[3, 2, 1], 10, "linear"],
        "funcion": dcg_at_k
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0], k=4, gain='linear'",
        "argumentos": [[0, 0, 0, 0], 4, "linear"],
        "funcion": dcg_at_k
    }
]

tabla_dcg_k = tabla_pruebas(
    "DCG@K",
    pruebas_dcg_k
)

,Caso,Entrada,Resultado
0,Ejemplo 1 - ganancia lineal,"[3, 2, 0, 1, 2], k=5, gain='linear'",5.466242
1,Ejemplo 2 - ganancia exponencial,"[3, 0, 2, 1, 3], k=5, gain='exponential'",11.638646
2,Vector vacío,"[], k=5, gain='linear'",0.000000
3,K mayor que longitud,"[3, 2, 1], k=10, gain='linear'",0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0], k=4, gain='linear'",0.000000


## NDCG at K

In [69]:
def ndcg_at_k(relevance_query, k, gain='exponential'):
    result = 0.0
    if len(relevance_query)>0 : 
        ideal_query = sorted(relevance_query, reverse=True)
        dcg_Ideal = dcg_at_k(ideal_query,k,gain)
        if dcg_Ideal > 0 :
            result = (dcg_at_k(relevance_query,k,gain))/(dcg_Ideal)
    return result    

### Example

In [70]:
pruebas_ndcg_k = [
    {
        "caso": "Ejemplo 1",
        "entrada": "[3, 2, 0, 1, 2], k=5, gain='linear'",
        "argumentos": [[3, 2, 0, 1, 2], 5],
        "funcion": ndcg_at_k
    },
    {
        "caso": "Ejemplo 2",
        "entrada": "[0, 1, 3, 2, 0], k=5, gain='linear'",
        "argumentos": [[0, 1, 3, 2, 0], 5],
        "funcion": ndcg_at_k
    },
    {
        "caso": "Vector vacío",
        "entrada": "[], k=5, gain='linear'",
        "argumentos": [[], 5, "linear"],
        "funcion": ndcg_at_k
    },
    {
        "caso": "K mayor que longitud",
        "entrada": "[3, 2, 1], k=10, gain='linear'",
        "argumentos": [[3, 2, 1], 10, "linear"],
        "funcion": ndcg_at_k
    },
    {
        "caso": "Sin documentos relevantes",
        "entrada": "[0, 0, 0, 0], k=4, gain='linear'",
        "argumentos": [[0, 0, 0, 0], 4, "linear"],
        "funcion": ndcg_at_k
    }
]

tabla_ndcg_k = tabla_pruebas(
    "NDCG@K",
    pruebas_ndcg_k
)

,Caso,Entrada,Resultado
0,Ejemplo 1,"[3, 2, 0, 1, 2], k=5, gain='linear'",0.968638
1,Ejemplo 2,"[0, 1, 3, 2, 0], k=5, gain='linear'",0.577353
2,Vector vacío,"[], k=5, gain='linear'",0.000000
3,K mayor que longitud,"[3, 2, 1], k=10, gain='linear'",0.000000
4,Sin documentos relevantes,"[0, 0, 0, 0], k=4, gain='linear'",0.000000


# Motores de Busqueda

## Procesamiento DataSet

### Lectura de documentos

el contenido empezaba con el titulo del documento, lo cual puede resultar en busquedas dobles en elementos donde solo esta una vez, por eso convienve filtrarlo desde la lectura

In [71]:
def read_naf_document(path):
    tree = ET.parse(path)
    root = tree.getroot()

    header = root.find("nafHeader")
    file_desc = header.find("fileDesc")
    public = header.find("public")
    raw = root.find("raw")
    titulo = file_desc.get("title")
    contenido = raw.text

    if titulo is not None:
        contenido = contenido[len(titulo):].lstrip(". \n")
        return {
            "id": public.get("publicId"),
            "titulo": titulo,
            "contenido": contenido}
    else:
        return {
            "id": public.get("publicId"),
            "contenido": contenido}

docs_path = Path("DataBases/docs-raw-texts")
naf_files = list(docs_path.glob("*.naf"))
documents = []

for file_path in naf_files:
    document = read_naf_document(file_path)
    documents.append(document)

### Pre-Procesamiento.Limpieza y normalizacion

In [72]:
def clean_text(text):
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_text(text, remove_accents=False):
    text = unicodedata.normalize("NFKC", text).lower()
    if remove_accents:
        text = "".join(
            char
            for char in unicodedata.normalize("NFD", text)
            if unicodedata.category(char) != "Mn"
        )
    return text
for document in documents:
    texto = document["titulo"] + " " + document["contenido"]
    texto = clean_text(texto)
    document["contenido_limpio"] = normalize_text(texto, True)

### Tokenizacion
Se seleccionó spaCy como tokenizer porque su segmentación permite separar términos unidos por signos de puntuación, como guiones, y conservar sus componentes como tokens independientes. Esto resulta conveniente para la recuperación booleana mediante índice invertido, ya que permite que términos individuales como jean o nicolas puedan ser indexados y recuperados independientemente.en la tabla se puede ver las evidencias de eso

In [73]:
def tokenize_nltk(text):
    return word_tokenize(text)

nlp = spacy.load("en_core_web_sm")
def tokenize_spacy(text):
    doc = nlp(text)
    return [token.text for token in doc]

In [ ]:
def compare_tokens(tokens_nltk, tokens_spacy):
    set_nltk = set(tokens_nltk)
    set_spacy = set(tokens_spacy)

    solo_nltk = [token for token in tokens_nltk if token not in set_spacy]
    solo_spacy = [token for token in tokens_spacy if token not in set_nltk]

    return solo_nltk, solo_spacy

resultados = []

for document in documents:
    texto = document["contenido_limpio"]

    tokens_nltk = tokenize_nltk(texto)
    tokens_spacy = tokenize_spacy(texto)

    set_nltk = set(tokens_nltk)
    set_spacy = set(tokens_spacy)

    solo_nltk = [token for token in tokens_nltk if token not in set_spacy]
    solo_spacy = [token for token in tokens_spacy if token not in set_nltk]

    resultados.append({
        "id": document["id"],
        "titulo": document["titulo"],
        "tokens_nltk": len(tokens_nltk),
        "tokens_spacy": len(tokens_spacy),
        "diferencia": len(tokens_nltk) - len(tokens_spacy),
        "solo_NLTK": solo_nltk,
        "solo_spaCy": solo_spacy
    })

comparacion = pd.DataFrame(resultados)

display(comparacion)

### Limpieza de tokens
Quitamos palabras de parada y puntuacion en los tockes 

In [ ]:
for document in documents:
    document["tokens"] = [ token for token in nlp(document["contenido_limpio"]) if not token.is_punct and not token.is_stop ]

### Steamming
Elegimos utilizar Porter Stemmer porque queremos que palabras que representan una misma idea, pero que aparecen en diferentes formas, puedan ser tratadas como un mismo término durante la búsqueda. Por ejemplo, engine y engines se convierten en engin, mientras que digestion y digestive se convierten en digest. Aunque el resultado no siempre sea una palabra real, esto no es un problema para nuestro motor, porque el stem se utiliza únicamente como término interno del índice. Así podemos aumentar las posibilidades de encontrar documentos relevantes aunque la consulta y el documento utilicen formas diferentes de una palabra.

In [ ]:
for document in documents:
    document["tokens"] = [stemmer.stem(token.text) for token in document["tokens"]]

In [ ]:
print("Número de documentos:", len(documents))
documentos_sin_tokens = [
    document["id"]
    for document in documents
    if not document["tokens"]
]

print("Documentos sin tokens:", documentos_sin_tokens)
cantidad_tokens = [
    len(document["tokens"])
    for document in documents
]

print("Mínimo:", min(cantidad_tokens))
print("Máximo:", max(cantidad_tokens))
print("Promedio:", sum(cantidad_tokens) / len(cantidad_tokens))

## Recuperación booleana usando índice invertido (BSII).

### Creacion indice inverso

In [ ]:
def build_inverted_index(documents):
    index = {}
    for document in documents:
        document_id = document["id"]
        for token in document["tokens"]:
            if token not in index:
                index[token] = set()
            index[token].add(document_id)
    for token in index:
        index[token] = sorted(index[token])
    return index

inverted_index = build_inverted_index(documents)
tabla_indice = pd.DataFrame(
    [
        {
            "termino": termino,
            "documentos": ", ".join(sorted(documentos))
        }
        for termino, documentos in list(inverted_index.items())[:20]
    ]
)

display(tabla_indice)

In [ ]:
memoria_indice = sys.getsizeof(inverted_index)

for termino, postings in inverted_index.items():
    memoria_indice += sys.getsizeof(termino)
    memoria_indice += sys.getsizeof(postings)

    for documento in postings:
        memoria_indice += sys.getsizeof(documento)

print("Número de documentos:", len(documents))
print("Tamaño del vocabulario:", len(inverted_index))
print("Memoria aproximada del índice:", memoria_indice, "bytes")

In [ ]:
termino_mas_frecuente = None
termino_menos_frecuente = None

max_postings = 0
min_postings = None

for termino, postings in inverted_index.items():
    cantidad_postings = len(postings)

    if cantidad_postings > max_postings:
        max_postings = cantidad_postings
        termino_mas_frecuente = termino

    if min_postings is None or cantidad_postings < min_postings:
        min_postings = cantidad_postings
        termino_menos_frecuente = termino

print("Término más frecuente:", termino_mas_frecuente)
print("Número de postings:", max_postings)

print("Término menos frecuente:", termino_menos_frecuente)
print("Número de postings:", min_postings)

### Consultas booleanas

Sin skip pointers, la intersección tiene complejidad O(m+n). Con skip pointers se pueden reducir las comparaciones y, en la práctica, procesar la intersección en menos tiempo al saltar segmentos que no pueden producir coincidencias; sin embargo, el beneficio depende de la distribución y longitud de las listas, por lo que no debe asumirse una mejora asintótica fija para todos los casos.

In [ ]:
def build_skip_pointers(postings):
    skips = {}
    skip_length = int(math.sqrt(len(postings)))
    if skip_length < 2:
        return skips

    for i in range(0, len(postings) - skip_length, skip_length):
        skips[i] = i + skip_length

    return skips

def intersect_without_skips(postings1, postings2):
    result = []
    comparisons = 0
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        comparisons += 1
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            i += 1
        else:
            j += 1
    return result, comparisons

def union_postings(postings1, postings2):
    result = []
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            result.append(postings1[i])
            i += 1
        else:
            result.append(postings2[j])
            j += 1
    while i < len(postings1):
        result.append(postings1[i])
        i += 1

    while j < len(postings2):
        result.append(postings2[j])
        j += 1

    return result

def not_postings(postings, documents):
    result = []
    documentos_postings = set(postings)
    for document in documents:
        if document["id"] not in documentos_postings:
            result.append(document["id"])
    result.sort()
    return result

def intersect_with_skips(postings1, postings2, skips1, skips2):
    result = []
    comparisons = 0
    skips_used = 0
    i = 0
    j = 0
    while i < len(postings1) and j < len(postings2):
        comparisons += 1
        if postings1[i] == postings2[j]:
            result.append(postings1[i])
            i += 1
            j += 1
        elif postings1[i] < postings2[j]:
            if (i in skips1 and postings1[skips1[i]] <= postings2[j]):
                i = skips1[i]
                skips_used += 1
            else:
                i += 1
        else:
            if (j in skips2 and postings2[skips2[j]] <= postings1[i]):
                j = skips2[j]
                skips_used += 1
            else:
                j += 1
    return result, comparisons, skips_used

### Evaluacion Queries

#### Consulta booleana

In [ ]:
def split_boolean_query(query):
    query = query.replace("(", " ( ")
    query = query.replace(")", " ) ")

    return query.split()

def process_boolean_tokens(tokens):
    result = []
    operators = ["AND", "OR", "NOT", "(", ")"]
    for token in tokens:
        if token in operators:
            result.append(token)
        else:
            content = clean_text(token)
            contentN = normalize_text(content, True)
            processed_tokens = [ t.text for t in nlp(contentN) if not t.is_punct and not t.is_stop]
            for processed_token in processed_tokens:result.append(stemmer.stem(processed_token))
    return result

def parse_not(tokens, position):
    if tokens[position] == "NOT":
        result, position = parse_not(tokens, position + 1)
        return not_postings(result, documents), position
    if tokens[position] == "(":
        result, position = parse_or(tokens, position + 1)
        return result, position + 1
    return inverted_index.get(tokens[position], []), position + 1

def parse_and(tokens, position):
    result, position = parse_not(tokens, position)
    while position < len(tokens) and tokens[position] == "AND":
        next_result, position = parse_not(tokens, position + 1)
        skips1 = build_skip_pointers(result)
        skips2 = build_skip_pointers(next_result)
        result, _, _ = intersect_with_skips(result, next_result, skips1, skips2)
    return result, position

def parse_or(tokens, position):
    result, position = parse_and(tokens, position)
    while position < len(tokens) and tokens[position] == "OR":
        next_result, position = parse_and(tokens, position + 1)
        result = union_postings(result, next_result)
    return result, position

def evaluate_boolean_query(tokens):
    result, _ = parse_or(tokens, 0)
    return result

In [ ]:
pruebas = [
    "river AND french",
    "river OR french",
    "NOT river",
    "river AND NOT french",
    "river OR NOT french",
    "river OR french AND method",
    "river AND (french OR method)",
    "(river OR french) AND method"
]

for query in pruebas:
    tokens = split_boolean_query(query)
    tokens = process_boolean_tokens(tokens)

    resultado = evaluate_boolean_query(tokens)

    print(query)
    print("Tokens:", tokens)
    print("Cantidad:", len(resultado))
    print("Resultado:", resultado)
    print()

#### Consultas binarias AND

In [ ]:
queries_path = Path("DataBases/queries-raw-texts")
query_files = list(queries_path.glob("*.naf"))

queries = []

for file_path in query_files:
    query = read_naf_document(file_path)
    queries.append(query)

In [ ]:
for query in queries:

    content = clean_text(query["contenido"])
    contentN = normalize_text( content,True)
    tokens = [
        token.text
        for token in nlp(contentN)
        if not token.is_punct and not token.is_stop
    ]
    query["stems"] = [ stemmer.stem(token) for token in tokens]

Los skip pointers no garantizan una reducción de comparaciones para todas las consultas. Su beneficio depende de la distribución de los términos en las posting lists y de si, durante la intersección, se alcanza una posición desde la cual el salto permita avanzar sin saltarse un posible resultado.

In [ ]:
resultados_queries = []

for query in queries:

    terms = query["stems"]

    if not terms:
        resultados_queries.append({
            "query": query["id"],
            "terminos": "",
            "resultado": [],
            "comparaciones_sin_skips": 0,
            "comparaciones_con_skips": 0,
            "skips_utilizados": 0,
            "iguales": True
        })
        continue

    resultado_sin = inverted_index.get(terms[0], [])
    comparaciones_sin = 0

    for term in terms[1:]:
        postings = inverted_index.get(term, [])
        resultado_sin, comparaciones = intersect_without_skips(
            resultado_sin,
            postings
        )
        comparaciones_sin += comparaciones

        if not resultado_sin:
            break

    resultado_con = inverted_index.get(terms[0], [])
    comparaciones_con = 0
    skips_utilizados = 0

    for term in terms[1:]:

        postings = inverted_index.get(term, [])

        skips_resultado = build_skip_pointers(resultado_con)
        skips_postings = build_skip_pointers(postings)

        resultado_con, comparaciones, skips = intersect_with_skips(
            resultado_con,
            postings,
            skips_resultado,
            skips_postings
        )

        comparaciones_con += comparaciones
        skips_utilizados += skips

        if not resultado_con:
            break

    resultados_queries.append({
        "query": query["id"],
        "terminos": " AND ".join(terms),
        "resultado": resultado_con,
        "comparaciones_sin_skips": comparaciones_sin,
        "comparaciones_con_skips": comparaciones_con,
        "skips_utilizados": skips_utilizados,
        "iguales": resultado_sin == resultado_con
    })

tabla_queries = pd.DataFrame(resultados_queries)

display(tabla_queries)

In [ ]:
tabla_queries_ordenada = tabla_queries.copy()
numeros = []

for query_id in tabla_queries_ordenada["query"]:
    numeros.append(int(query_id[1:]))

tabla_queries_ordenada["numero_query"] = numeros
tabla_queries_ordenada = tabla_queries_ordenada.sort_values("numero_query")
output_path = Path("BSII-AND-queries_results")

with output_path.open("w", encoding="utf-8") as output_file:
    for _, row in tabla_queries_ordenada.iterrows():
        documents_found = sorted(row["resultado"])
        output_file.write(
            f'{row["query"]} {",".join(documents_found)}\n'
        )

print(f"Archivo generado: {output_path.resolve()}")

## Recuperación ranqueada y vectorización de documentos (RRDV)

### Representación vectorial ponderada tf.idf

La estrategia utiliza el índice invertido para obtener el DF de cada término y limitar el cálculo del TF a los documentos en los que el término aparece. Esto evita calcular el TF para documentos en los que el término no está presente y aprovecha la estructura construida para BSII. Sin embargo, el índice invertido almacena únicamente los identificadores de los documentos, por lo que la frecuencia del término debe calcularse nuevamente a partir de los tokens de cada documento. Además, para localizar cada documento asociado a un posting, la implementación recorre la colección de documentos. Por esta razón, aunque se reduce el número de documentos sobre los que se calcula el TF, la implementación no es completamente eficiente

In [ ]:
def build_tfidf_matrix(inverted_index, documents):
    tf_idf_matrix = {}

    for term in inverted_index:
        tf_idf_matrix[term] = {}
        df = len(inverted_index[term])
        idf = np.log10(len(documents) / df)

        for document_id in inverted_index[term]:
            for document in documents:
                if document["id"] == document_id:
                    tf = document["tokens"].count(term)
                    tf_idf_matrix[term][document_id] = np.log10(1 + tf) * idf
                    break

    return tf_idf_matrix     

 
tf_idf_matrix = build_tfidf_matrix(inverted_index, documents)

In [ ]:
def build_document_vectors(tf_idf_matrix, documents):

    document_vectors = {}

    for document in documents:
        document_id = document["id"]
        vector = []
        for term in tf_idf_matrix:
            vector.append(tf_idf_matrix[term].get(document_id, 0))
        document_vectors[document_id] = vector

    return document_vectors

document_vectors = build_document_vectors(tf_idf_matrix,documents)

def build_query_vector(query, tf_idf_matrix, inverted_index, documents):
    query_vector = []
    for term in tf_idf_matrix:
        tf = query["stems"].count(term)
        if tf > 0:
            df = len(inverted_index[term])
            idf = np.log10(len(documents) / df)
            tf_idf = np.log10(1 + tf) * idf
        else:
            tf_idf = 0
        query_vector.append(tf_idf)
    return query_vector
    

# vector ara uno de los queries
query_vector = build_query_vector(
    query,
    tf_idf_matrix,
    inverted_index,
    documents
)
print("Dimensión:", len(query_vector))
print("Primeros 10 valores:", query_vector[:10])

    

In [ ]:

ids = list(document_vectors.keys())[:10]

# Terminos que tienen algun peso en esos documentos
terms = []

for term in tf_idf_matrix:
    if any(tf_idf_matrix[term].get(doc_id, 0) > 0 for doc_id in ids):
        terms.append(term)

# Construimos la tabla
vector_table = pd.DataFrame(
    {
        term: [tf_idf_matrix[term].get(doc_id, 0) for doc_id in ids]
        for term in terms
    },
    index=ids
)

vector_table = vector_table.T
display(vector_table)

### Similitud del coseno

In [ ]:
def cosine_similarity(vector1, vector2):
    dot_product = np.dot(vector1, vector2)
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot_product / (norm1 * norm2)

d1 = document_vectors["d096"]
d2 = document_vectors["d169"]

similarity = cosine_similarity(d1, d2)

print("Similitud:", similarity)

In [ ]:
def rank_documents(query,document_vectors,tf_idf_matrix,inverted_index,documents):

    query_vector = build_query_vector(query,tf_idf_matrix,inverted_index,documents)
    rankings = []

    for document_id, document_vector in document_vectors.items():

        similarity = cosine_similarity(query_vector,document_vector)

        if similarity > 0:
            rankings.append((document_id, similarity))

    return sorted(rankings,key=lambda x: x[1],reverse=True)

In [ ]:
coseno_rankings = {}

for query in queries:
    coseno_rankings[query["id"]] = rank_documents(query,document_vectors,tf_idf_matrix,inverted_index,documents)

In [ ]:
filas = []

for query_id, ranking in coseno_rankings.items():

    for posicion, (document_id, similarity) in enumerate(ranking[:10], start=1):

        filas.append({
            "query": query_id,
            "posición": posicion,
            "documento": document_id,
            "similitud": similarity
        })

tabla_ranking = pd.DataFrame(filas)

display(tabla_ranking)

In [ ]:
query_id = "q06"

tabla_q06 = pd.DataFrame(
    coseno_rankings[query_id][:10],
    columns=["documento", "similitud"]
)

tabla_q06.insert(0, "posición", range(1, len(tabla_q06) + 1))

display(tabla_q06)

### BM25

In [ ]:
corpus = [document["tokens"] for document in documents]

bm25 = BM25Okapi(
    corpus,
    k1=1.5,
    b=0.75
)
bm25_rankings = {}
filas = []
for query in queries:
    scores = bm25.get_scores(query["stems"])
    bm25_rankings[query["id"]] = [(documents[i]["id"], scores[i]) for i in np.argsort(scores)[::-1] if scores[i] > 0]

for query_id, ranking in bm25_rankings.items():

    for posicion, (document_id, score) in enumerate(ranking[:10], start=1):

        filas.append({
            "query": query_id,
            "posición": posicion,
            "documento": document_id,
            "BM25": score
        })

tabla_bm25 = pd.DataFrame(filas)

display(tabla_bm25)

Al variar b entre 0 y 1 se observa que la normalización por longitud afecta el ranking de BM25. Para q06, con b=0, d297 ocupa el primer lugar y d329 el segundo; al aumentar b hasta 0.50, sus posiciones se intercambian y d329 permanece primero hasta b=1. Otros documentos, como d026, mantienen su posición durante toda la variación. Esto muestra que el efecto de b depende de las características de cada documento y que una mayor normalización por longitud puede modificar el orden de documentos con puntajes similares.

In [ ]:
query = next(q for q in queries if q["id"] == "q06")

resultados_bm25 = []

for b in [0, 0.25, 0.5, 0.75, 1]:

    bm25 = BM25Okapi(
        corpus,
        k1=1.5,
        b=b
    )

    scores = bm25.get_scores(query["stems"])
    ranking = [(documents[i]["id"], scores[i])for i in np.argsort(scores)[::-1] if scores[i] > 0]

    for posicion, (documento, score) in enumerate(ranking[:10], 1):

        resultados_bm25.append({
            "b": b,
            "posición": posicion,
            "documento": documento,
            "score": score
        })


In [ ]:
tabla_bm25 = pd.DataFrame(resultados_bm25)

tabla_bm25["resultado"] = (
    tabla_bm25["documento"] 
    + " (" 
    + tabla_bm25["score"].round(2).astype(str) 
    + ")"
)

tabla_bm25 = tabla_bm25.pivot(
    index="posición",
    columns="b",
    values="resultado"
)

display(tabla_bm25)

### Creacion de archivos de resultados

In [ ]:
with open("RRDV-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query in queries:

        resultados = rank_documents(query,document_vectors,tf_idf_matrix, inverted_index,documents)

        resultados_formateados = [ f"{document_id}: {similitud:.6f}"  for document_id, similitud in resultados ]

        archivo.write(
            f"{query['id']} {','.join(resultados_formateados)}\n"
        )

In [ ]:
with open("BM25-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query_id, ranking in bm25_rankings.items():

        resultados = [f"{document_id}: {score:.6f}" for document_id, score in ranking]

        archivo.write(
            f"{query_id} {','.join(resultados)}\n"
        )

### Evaluacion de metricas (Resultados)

In [ ]:
relevance_judgments = {}

with open("DataBases/relevance-judgments.tsv", "r", encoding="utf-8") as archivo:
    for linea in archivo:
        query_id, documentos = linea.strip().split("\t")
        relevance_judgments[query_id] = {}
        for documento in documentos.split(","):
            document_id, relevancia = documento.split(":")
            relevance_judgments[query_id][document_id] = int(relevancia)

In [ ]:
def evaluate_ranking(ranking, relevance_judgments):

    resultados = []

    for query_id, ranking_query in ranking.items():
        juicios = relevance_judgments.get(query_id, {})
        K = len(juicios)
        documentos = [document_id for document_id, score in ranking_query]
        relevancia_binaria = [1 if document_id in juicios else 0 for document_id in documentos]
        relevancia_graduada = [ juicios.get(document_id, 0) for document_id in documentos]

        resultados.append({
            "query": query_id,
            "K": K,
            "P@K": precision_at_k(relevancia_binaria, K),
            "R@K": recall_at_k(relevancia_binaria, K, K),
            "NDCG@K": ndcg_at_k(relevancia_graduada, K, "linear"),
        })

    return pd.DataFrame(resultados)

In [ ]:
evaluacion_rrdv = evaluate_ranking(coseno_rankings,relevance_judgments)
evaluacion_bm25 = evaluate_ranking(bm25_rankings,relevance_judgments)

In [ ]:
map_cos = mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in coseno_rankings.items()
    ]
)

map_bm25 = mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in bm25_rankings.items()
    ]
)

print("MAP COS:", map_cos)
print("MAP BM25:", map_bm25)

## Recuperación ranqueada con GENSIM

In [ ]:
corpus = [document["tokens"] for document in documents]
dictionary = corpora.Dictionary(corpus)

corpus_bow = [dictionary.doc2bow(document["tokens"])for document in documents]

print("Tamaño del vocabulario - GENSIM:", len(dictionary))

tfidf = models.TfidfModel(corpus_bow,id2word=dictionary,smartirs="ltc")
corpus_tfidf = tfidf[corpus_bow]
index = similarities.SparseMatrixSimilarity(corpus_tfidf,num_features=len(dictionary),num_best=None)

gensim_rankings = {}

for query in queries:
    query_bow = dictionary.doc2bow(query["stems"])
    query_tfidf = tfidf[query_bow]
    scores = index[query_tfidf]
    ranking = [(documents[i]["id"], scores[i]) for i in np.argsort(scores)[::-1] if scores[i] > 0]
    gensim_rankings[query["id"]] = ranking

In [ ]:
with open("GENSIM-consultas_resultados", "w", encoding="utf-8") as archivo:

    for query_id, ranking in gensim_rankings.items():

        resultados = [
            f"{document_id}: {score:.6f}"
            for document_id, score in ranking
        ]

        archivo.write(
            f"{query_id} {','.join(resultados)}\n"
        )

In [ ]:
evaluacion_gensim = evaluate_ranking(gensim_rankings,relevance_judgments)

In [ ]:
map_gensim = mean_average_precision(
    [[1 if document_id in relevance_judgments[query_id] else 0 for document_id, score in ranking]
        for query_id, ranking in gensim_rankings.items()
    ]
)
print("MAP GENSIM:", map_gensim)

## Comparativa de metricas

#### Tabla Comparativa metricas

In [ ]:

evaluacion_comparativa = evaluacion_rrdv.merge(
    evaluacion_bm25,
    on=["query", "K"],
    suffixes=("_COS", "_BM25")
)

evaluacion_comparativa = evaluacion_comparativa.merge(
    evaluacion_gensim,
    on=["query", "K"]
)

evaluacion_comparativa = evaluacion_comparativa.rename(
    columns={
        "P@K": "P@K_GENSIM",
        "R@K": "R@K_GENSIM",
        "NDCG@K": "NDCG@K_GENSIM"
    }
)

evaluacion_comparativa["numero_query"] = (
    evaluacion_comparativa["query"].str[1:].astype(int)
)

evaluacion_comparativa = evaluacion_comparativa.sort_values(
    "numero_query"
).drop(columns="numero_query")

evaluacion_comparativa = evaluacion_comparativa[
    [
        "query",
        "K",
        "P@K_COS",
        "P@K_BM25",
        "P@K_GENSIM",
        "R@K_COS",
        "R@K_BM25",
        "R@K_GENSIM",
        "NDCG@K_COS",
        "NDCG@K_BM25",
        "NDCG@K_GENSIM"
    ]
]

display(evaluacion_comparativa)

#### Tabla Comparativa MPA

In [ ]:
tabla_map = pd.DataFrame({
    "Función de ranqueo": [
        "COS",
        "BM25",
        "GENSIM"
    ],
    "MAP": [
        map_cos,
        map_bm25,
        map_gensim
    ]
})

display(tabla_map)